In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 34.6 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import RagTokenizer, RagSequenceForGeneration

In [ ]:
# Step 1: Example documents
documents = [
    "Meditation helps reduce stress and anxiety.",
    "Regular meditation can improve concentration.",
    "Exercise is beneficial for physical health.",
    "Meditation also promotes emotional health."
]

# Step 2: Embed documents
embedder = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = embedder.encode(documents)

# Step 3: Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

# Step 4: Prepare query and embed
query = "What benefits does meditation have?"
# query = "partying in los vegas"
query_embedding = embedder.encode([query])

# Search top 2 documents
k = 2
distances, indices = index.search(query_embedding, k)

# Retrieve top docs
retrieved_docs = [documents[idx] for idx in indices[0]]

print("Retrieved documents:")
for doc in retrieved_docs:
    print("-", doc)


Retrieved documents:
- Meditation helps reduce stress and anxiety.
- Meditation also promotes emotional health.


In [ ]:
distances

array([[0.46085522, 0.5352076 ]], dtype=float32)

In [ ]:
import os


# Set other API keys similarly
os.environ["HF_TOKEN"] = "REMOVED_HUGGING_FACE_TOKEN"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import torch
import faiss
import numpy as np

# 1. Load Mistral tokenizer & model
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

# 2. Sample documents
documents = [
    "Meditation helps reduce stress and anxiety.",
    "Regular meditation can improve concentration.",
    "Exercise is beneficial for physical health.",
    "Meditation also promotes emotional health."
]

# 3. Use sentence-transformers for dense embeddings
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedder.encode(documents, convert_to_numpy=True)

# 4. Create FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

# 5. Define query and retrieve top documents
query = "What are the benefits of meditation?"
query_embedding = embedder.encode([query])
_, indices = index.search(np.array(query_embedding), k=2)

retrieved_docs = [documents[i] for i in indices[0]]

# 6. Concatenate retrieved docs with query
context = "\n".join(retrieved_docs)
prompt = f"""Answer the question based on the context below.

Context:
{context}

Question: {query}
Answer:"""

# 7. Tokenize & generate
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
outputs = model.generate(input_ids, max_new_tokens=100)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Answer the question based on the context below.

Context:
Meditation helps reduce stress and anxiety.
Meditation also promotes emotional health.

Question: What are the benefits of meditation?
Answer: The benefits of meditation include reducing stress and anxiety, as well as promoting emotional health.


In [ ]:
# # Step 5: Use Hugging Face RAG for generation (context concatenated manually here)
# from sentence_transformers import SentenceTransformer
# import faiss
# import numpy as np
# from transformers import RagTokenizer, RagSequenceForGeneration, RagRetriever

# # Step 1: Your knowledge base
# documents = [
#     "Meditation helps reduce stress and anxiety.",
#     "Regular meditation can improve concentration.",
#     "Exercise is beneficial for physical health.",
#     "Meditation also promotes emotional health."
# ]

# # Step 2: Sentence embeddings
# embedder = SentenceTransformer('all-MiniLM-L6-v2')
# doc_embeddings = embedder.encode(documents).astype(np.float32)

# # Step 3: Build FAISS index
# dimension = doc_embeddings.shape[1]
# index = faiss.IndexFlatL2(dimension)
# index.add(doc_embeddings)

# # Step 4: User query
# query = "What benefits does meditation have?"
# query_embedding = embedder.encode([query]).astype(np.float32)

# # Step 5: Retrieve top-k documents
# k = 2
# _, indices = index.search(query_embedding, k)
# retrieved_docs = [documents[i] for i in indices[0]]

# print("Retrieved Documents:")
# for doc in retrieved_docs:
#     print("-", doc)

# # Step 6: Load model and tokenizer
# tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-base")
# model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-base")

# # Step 7: Required hack: set dummy retriever to prevent internal assertion failure
# retriever = RagRetriever.from_pretrained("facebook/rag-token-base", index_name="custom", passages_path=None)
# model.set_retriever(retriever)

# # Step 8: Tokenize query and contexts
# question_inputs = tokenizer(query, return_tensors="pt")
# context_inputs = tokenizer(retrieved_docs, return_tensors="pt", padding=True, truncation=True)

# # Step 9: Generate answer with custom context
# outputs = model.generate(
#     input_ids=question_inputs["input_ids"],
#     attention_mask=question_inputs["attention_mask"],
#     context_input_ids=context_inputs["input_ids"],
#     context_attention_mask=context_inputs["attention_mask"],
#     n_docs=k,
#     num_beams=4,
#     max_length=100
# )

# # Step 10: Decode the output
# answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# print("\nGenerated Answer:")
# print(answer)


In [ ]:
# from transformers import RagTokenizer, RagSequenceForGeneration

# # Load model and tokenizer
# tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-base")
# model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-base")

# # Sample input: query + retrieved context
# query = "What are the benefits of meditation?"
# context = "Meditation helps reduce stress. It improves focus and emotional well-being."

# # Option 1: Pass just the query (RAG internally handles retrieval)
# inputs = tokenizer(query, return_tensors="pt")

# # Option 2: If you manually add context, concatenate carefully
# # inputs = tokenizer(query + " " + context, return_tensors="pt", truncation=True)

# # Check inputs
# print("Input IDs:", inputs.get("input_ids"))
# print("Attention Mask:", inputs.get("attention_mask"))

# # Make sure neither is None before generating
# if inputs.get("input_ids") is not None and inputs.get("attention_mask") is not None:
#     outputs = model.generate(
#         input_ids=inputs["input_ids"],
#         attention_mask=inputs["attention_mask"]
#     )
#     answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
#     print("Answer:", answer)
# else:
#     print("Tokenizer failed to produce valid input.")


In [ ]:
# inputs = tokenizer(query, return_tensors="pt")

# print("Inputs:", inputs)
# print("Input IDs:", inputs.get("input_ids"))
# print("Attention Mask:", inputs.get("attention_mask"))

In [ ]:
# from transformers import RagTokenizer, RagSequenceForGeneration

# tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-base")
# model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-base")

# query = "What is the capital of France?"
# inputs = tokenizer(query, return_tensors="pt")

# outputs = model.generate(inputs["input_ids"])
# answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# print("Answer:", answer)


In [ ]:
# from sentence_transformers import SentenceTransformer
# import faiss
# import numpy as np
# from transformers import RagTokenizer, RagSequenceForGeneration

# # Step 1: Example documents
# documents = [
#     "Meditation helps reduce stress and anxiety.",
#     "Regular meditation can improve concentration.",
#     "Exercise is beneficial for physical health.",
#     "Meditation also promotes emotional health."
# ]

# # Step 2: Embed documents
# embedder = SentenceTransformer('all-MiniLM-L6-v2')
# doc_embeddings = embedder.encode(documents).astype(np.float32)

# # Step 3: Build FAISS index
# dimension = doc_embeddings.shape[1]
# index = faiss.IndexFlatL2(dimension)
# index.add(doc_embeddings)

# # Step 4: Prepare query and embed
# query = "What benefits does meditation have?"
# query_embedding = embedder.encode([query]).astype(np.float32)

# # Search top 2 documents
# k = 2
# distances, indices = index.search(query_embedding, k)

# # Retrieve top docs
# retrieved_docs = [documents[idx] for idx in indices[0]]

# print("Retrieved documents:")
# for doc in retrieved_docs:
#     print("-", doc)

# # --- Now tokenize query and retrieved docs separately ---

# tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-base")
# model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-base")

# # Tokenize the question
# question_inputs = tokenizer([query], return_tensors="pt")

# # Tokenize the retrieved documents as context (pass a batch of docs)
# context_inputs = tokenizer(retrieved_docs, padding=True, truncation=True, return_tensors="pt")

# # Step 5: Generate answer using the question and retrieved contexts
# outputs = model.generate(
#     input_ids=question_inputs["input_ids"],
#     attention_mask=question_inputs["attention_mask"],
#     context_input_ids=context_inputs["input_ids"],
#     context_attention_mask=context_inputs["attention_mask"],
#     n_docs=k,  # number of retrieved docs
#     num_beams=2,
#     max_length=100,
# )

# answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# print("\nGenerated answer:")
# print(answer)


In [ ]:
# from sentence_transformers import SentenceTransformer
# import faiss
# import numpy as np
# from transformers import RagTokenizer, RagSequenceForGeneration, RagRetriever

# documents = [
#     "Meditation helps reduce stress and anxiety.",
#     "Regular meditation can improve concentration.",
#     "Exercise is beneficial for physical health.",
#     "Meditation also promotes emotional health."
# ]

# embedder = SentenceTransformer('all-MiniLM-L6-v2')
# doc_embeddings = embedder.encode(documents).astype(np.float32)

# dimension = doc_embeddings.shape[1]
# index = faiss.IndexFlatL2(dimension)
# index.add(doc_embeddings)

# query = "What benefits does meditation have?"
# query_embedding = embedder.encode([query]).astype(np.float32)

# k = 2
# distances, indices = index.search(query_embedding, k)

# retrieved_docs = [documents[idx] for idx in indices[0]]

# print("Retrieved documents:")
# for doc in retrieved_docs:
#     print("-", doc)

# tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-base")
# model = RagSequenceForGeneration.from_pretrained("facebook/rag-token-base")

# # Set a dummy retriever so model.generate accepts context inputs
# retriever = RagRetriever.from_pretrained("facebook/rag-token-base", index_name="custom", passages_path=None)
# model.set_retriever(retriever)

# question_inputs = tokenizer([query], return_tensors="pt")
# context_inputs = tokenizer(retrieved_docs, padding=True, truncation=True, return_tensors="pt")

# outputs = model.generate(
#     input_ids=question_inputs["input_ids"],
#     attention_mask=question_inputs["attention_mask"],
#     context_input_ids=context_inputs["input_ids"],
#     context_attention_mask=context_inputs["attention_mask"],
#     n_docs=k,
#     num_beams=2,
#     max_length=100,
# )

# answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
# print("\nGenerated answer:")
# print(answer)
